In [4]:
# pip install scikit-learn
# So two characters are close if they tend to 
# appear with the same lexical theme words, in the same tomes, with similar frequencies.

import pandas as pd
import re
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_distances
from sklearn.manifold import MDS

# =========================
# FILES
# =========================
corpus_file = "La-ballade-de-Pern-intégrale-sentences-lemma.csv"
characters_file = "Noms personnages de Pern.ods"

# =========================
# PARAMETERS
# =========================
TARGET_WORDS = ["amour", "aimer", "couple"]

TEXT_COL = "text_lemma"
BOOK_COL = "book"

# graph parameters
MAX_DISTANCE_FOR_EDGE = 0.75   # lower = stricter graph
MIN_TOTAL_THEME_COUNT = 1      # exclude characters with 0 theme counts
FIGSIZE = (12, 10)

# =========================
# LOAD CHARACTERS
# =========================
sheet = pd.read_excel(
    characters_file,
    sheet_name="Nom et genre persos",
    engine="odf"
)

characters = set()
for col in ["NOMS HUMAINS", "NOMS DRAGONS"]:
    if col in sheet.columns:
        characters.update(sheet[col].dropna().astype(str).str.strip())

characters = sorted(c for c in characters if c)

print("Number of characters:", len(characters))

# =========================
# LOAD CORPUS
# =========================
df = pd.read_csv(corpus_file)

if TEXT_COL not in df.columns:
    raise ValueError(f"Column '{TEXT_COL}' is required.")

if BOOK_COL not in df.columns:
    raise ValueError(f"Column '{BOOK_COL}' is required.")

df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
df[BOOK_COL] = df[BOOK_COL].fillna("UNKNOWN").astype(str)

# =========================
# REGEX
# =========================
target_patterns = {
    w: re.compile(r"\b" + re.escape(w) + r"\b", re.IGNORECASE)
    for w in TARGET_WORDS
}

character_patterns = {
    char: re.compile(r"\b" + re.escape(char) + r"\b", re.IGNORECASE)
    for char in characters
}

# =========================
# COUNT CHARACTER × BOOK × TARGET WORD
# =========================
records = []

for _, row in df.iterrows():
    text = row[TEXT_COL]
    book = row[BOOK_COL]

    # count target words in this sentence
    word_counts = {}
    total_targets = 0
    for w, pat in target_patterns.items():
        c = len(pat.findall(text))
        word_counts[w] = c
        total_targets += c

    if total_targets == 0:
        continue

    # detect characters in sentence
    chars_in_sentence = [
        char for char, pat in character_patterns.items()
        if pat.search(text)
    ]

    if not chars_in_sentence:
        continue

    for char in chars_in_sentence:
        for w, c in word_counts.items():
            if c > 0:
                records.append({
                    "character": char,
                    "book": book,
                    "target_word": w,
                    "count": c
                })

cooc_long = pd.DataFrame(records)

if cooc_long.empty:
    raise ValueError("No character/theme co-occurrences found.")

# =========================
# BUILD CHARACTER PROFILES
# =========================
# profile dimensions = book__word
cooc_long["feature"] = cooc_long["book"].astype(str) + "__" + cooc_long["target_word"].astype(str)

profile_table = cooc_long.pivot_table(
    index="character",
    columns="feature",
    values="count",
    aggfunc="sum",
    fill_value=0
)

# optional: keep only characters with at least some theme signal
profile_table["TOTAL"] = profile_table.sum(axis=1)
profile_table = profile_table[profile_table["TOTAL"] >= MIN_TOTAL_THEME_COUNT].copy()

totals = profile_table["TOTAL"].copy()
profile_table = profile_table.drop(columns="TOTAL")

print("Characters kept for distance analysis:", profile_table.shape[0])
print("Number of profile dimensions:", profile_table.shape[1])

# save profile table
profile_out = profile_table.copy()
profile_out.insert(0, "TOTAL", totals)
profile_out.to_csv("lexical_theme_character_profiles.csv", encoding="utf-8-sig")

# =========================
# DISTANCE MATRIX
# =========================
X = profile_table.values
chars = profile_table.index.tolist()

dist_matrix = cosine_distances(X)
dist_df = pd.DataFrame(dist_matrix, index=chars, columns=chars)
dist_df.to_csv("lexical_theme_distance_matrix.csv", encoding="utf-8-sig")

print("Created: lexical_theme_distance_matrix.csv")
print("Created: lexical_theme_character_profiles.csv")

# =========================
# EDGE LIST FOR NETWORK
# =========================
edge_rows = []
for i in range(len(chars)):
    for j in range(i + 1, len(chars)):
        d = dist_matrix[i, j]
        if d <= MAX_DISTANCE_FOR_EDGE:
            edge_rows.append({
                "Source": chars[i],
                "Target": chars[j],
                "Distance": d,
                "Weight_similarity": 1 - d
            })

edges_df = pd.DataFrame(edge_rows)
edges_df.to_csv("lexical_theme_distance_edges.csv", index=False, encoding="utf-8-sig")
print("Created: lexical_theme_distance_edges.csv")

# =========================
# NETWORK GRAPH
# =========================
G = nx.Graph()

for char in chars:
    G.add_node(char, total=int(totals.loc[char]))

for _, row in edges_df.iterrows():
    G.add_edge(
        row["Source"],
        row["Target"],
        weight=row["Weight_similarity"],
        distance=row["Distance"]
    )

plt.figure(figsize=FIGSIZE)

if len(G.nodes) > 0:
    pos = nx.spring_layout(G, weight="weight", seed=42)

    node_sizes = [300 + 40 * G.nodes[n]["total"] for n in G.nodes()]
    edge_widths = [1 + 4 * G[u][v]["weight"] for u, v in G.edges()]

    nx.draw_networkx_nodes(G, pos, node_size=node_sizes)
    nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.5)
    nx.draw_networkx_labels(G, pos, font_size=8)

    plt.title("Character distance graph based on lexical theme")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig("lexical_theme_distance_graph.png", dpi=300, bbox_inches="tight")
    plt.close()

    print("Created: lexical_theme_distance_graph.png")
else:
    print("No edges passed the distance threshold; graph not drawn.")

# =========================
# 2D DISTANCE MAP (MDS)
# =========================
if len(chars) >= 2:
    mds = MDS(
        n_components=2,
        dissimilarity="precomputed",
        random_state=42
    )
    coords = mds.fit_transform(dist_matrix)

    coords_df = pd.DataFrame(coords, columns=["Dim1", "Dim2"])
    coords_df.insert(0, "character", chars)
    coords_df.insert(1, "TOTAL", [totals.loc[c] for c in chars])
    coords_df.to_csv("lexical_theme_mds_coordinates.csv", index=False, encoding="utf-8-sig")

    plt.figure(figsize=FIGSIZE)
    plt.scatter(coords_df["Dim1"], coords_df["Dim2"], s=30 + coords_df["TOTAL"] * 10)

    for _, row in coords_df.iterrows():
        plt.text(row["Dim1"], row["Dim2"], row["character"], fontsize=8)

    plt.title("Character distance map (MDS) based on lexical theme")
    plt.xlabel("Dim1")
    plt.ylabel("Dim2")
    plt.tight_layout()
    plt.savefig("lexical_theme_mds_map.png", dpi=300, bbox_inches="tight")
    plt.close()

    print("Created: lexical_theme_mds_coordinates.csv")
    print("Created: lexical_theme_mds_map.png")

Number of characters: 492
Characters kept for distance analysis: 106
Number of profile dimensions: 36
Created: lexical_theme_distance_matrix.csv
Created: lexical_theme_character_profiles.csv
Created: lexical_theme_distance_edges.csv
Created: lexical_theme_distance_graph.png


/home/marc-at/.local/lib/python3.10/site-packages/sklearn/manifold/_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(


Created: lexical_theme_mds_coordinates.csv
Created: lexical_theme_mds_map.png
